# 05-2. 교차 검증과 그리드 서치

와인 분류 결정 트리의 성능을 검증 세트와 교차 검증으로 평가하고, `GridSearchCV`와 `RandomizedSearchCV`로 하이퍼파라미터를 선택한다.

## 학습 목표

- 훈련·검증·테스트 세트의 역할 구분
- 교차 검증이 한 번의 검증 세트보다 안정적인 이유 이해
- 그리드 서치와 랜덤 서치의 차이 이해
- 최적 모델을 고른 뒤 테스트 세트를 마지막에 평가
- `splitter='random'`의 의미 확인

## 1. 훈련·검증·테스트 세트

전체 데이터에서 테스트 세트를 먼저 떼어 놓고, 남은 훈련 세트를 다시 훈련용과 검증용으로 나눈다.

```text
전체 데이터
├─ 훈련 세트 80%
│  ├─ 훈련용 세트 80%
│  └─ 검증 세트 20%
└─ 테스트 세트 20%
```

검증 세트는 모델과 하이퍼파라미터를 고르는 데 사용하고, 테스트 세트는 선택이 끝난 뒤 마지막에 한 번 사용한다.

> **이전 질문과 연결 — “테스트 정확도가 높으면 굳이 모델을 수정할 필요가 없지 않나?”**  
> 테스트 세트를 반복해서 보며 모델을 고르면 테스트 세트에도 맞춘 셈이 된다. 그래서 수정 과정에는 검증 세트나 교차 검증을 사용하고, 테스트 점수는 최종 일반화 성능 확인용으로 남겨 둔다.

In [2]:
import pandas as pd

# 5-1에서 사용한 와인 데이터를 다시 불러온다.
wine = pd.read_csv('https://bit.ly/wine_csv_data')

In [3]:
# 모델의 입력 특성과 정답 타깃을 분리한다.
data = wine[['alcohol', 'sugar', 'pH']]
target = wine['class']

In [4]:
from sklearn.model_selection import train_test_split

# 전체 데이터의 20%를 최종 테스트 세트로 먼저 분리한다.
# 테스트 세트는 하이퍼파라미터 선택에 사용하지 않고 마지막에 한 번만 평가한다.
train_input, test_input, train_target, test_target = train_test_split(
    data,
    target,
    test_size=0.2,
    random_state=42
)

In [5]:
# 훈련 세트의 20%를 검증 세트로 다시 분리한다.
# sub_input은 실제 학습용, val_input은 모델 선택용이다.
sub_input, val_input, sub_target, val_target = train_test_split(
    train_input,
    train_target,
    test_size=0.2,
    random_state=42
)

In [6]:
# 훈련용 세트와 검증 세트의 크기를 확인한다.
# 전체 훈련 세트 5197개가 4157개와 1040개로 나뉜다.
print(sub_input.shape, val_input.shape)

(4157, 3) (1040, 3)


In [7]:
from sklearn.tree import DecisionTreeClassifier

# 훈련용 세트로 결정 트리를 학습한다.
dt = DecisionTreeClassifier(random_state=42)
dt.fit(sub_input, sub_target)

# 훈련용 세트와 검증 세트의 정확도를 비교한다.
print(dt.score(sub_input, sub_target))
print(dt.score(val_input, val_target))

0.9971133028626413
0.864423076923077


훈련용 정확도는 `0.9971`, 검증 정확도는 `0.8644`다. 제한 없는 결정 트리가 훈련 데이터를 거의 외운 상태라는 점은 5-1과 같다.

## 2. 교차 검증

검증 세트를 한 번만 나누면 어떤 샘플이 들어갔는지에 따라 점수가 달라질 수 있다. 교차 검증은 훈련 세트를 여러 조각으로 나누고 검증 역할을 번갈아 맡긴다.

```text
1회: [검증][훈련][훈련][훈련][훈련]
2회: [훈련][검증][훈련][훈련][훈련]
...
5회: [훈련][훈련][훈련][훈련][검증]
```

각 폴드의 검증 점수를 평균해 모델의 성능을 판단한다.

In [8]:
from sklearn.model_selection import cross_validate

# 기본 5-폴드 교차 검증을 수행한다.
# 매 폴드마다 훈련과 검증을 반복하고 시간과 검증 점수를 반환한다.
scores = cross_validate(dt, train_input, train_target)
print(scores)

{'fit_time': array([0.01774073, 0.03308392, 0.0312202 , 0.03809428, 0.01481485]), 'score_time': array([0.0112493 , 0.00780725, 0.00374126, 0.00350451, 0.00404   ]), 'test_score': array([0.86923077, 0.84615385, 0.87680462, 0.84889317, 0.83541867])}


In [9]:
import numpy as np

# 5개 폴드의 검증 정확도 평균을 계산한다.
# 여기의 test_score는 최종 test_input 점수가 아니라 각 폴드의 검증 점수이다.
print(np.mean(scores['test_score']))

0.855300214703487


In [10]:
from sklearn.model_selection import StratifiedKFold

# 분류 문제에서는 클래스 비율을 유지하는 StratifiedKFold를 사용한다.
# 기본 5-폴드 설정이라 앞의 cross_validate 결과와 같다.
scores = cross_validate(
    dt,
    train_input,
    train_target,
    cv=StratifiedKFold()
)
print(np.mean(scores['test_score']))

0.855300214703487


In [11]:
# 데이터를 섞은 뒤 10개 폴드로 나누는 분할기를 만든다.
splitter = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

# 10-폴드 교차 검증의 평균 정확도를 확인한다.
scores = cross_validate(
    dt,
    train_input,
    train_target,
    cv=splitter
)
print(np.mean(scores['test_score']))

0.8574181117533719


- 기본 5-폴드 평균: `0.8553`
- 명시적 `StratifiedKFold` 5-폴드 평균: `0.8553`
- 섞은 10-폴드 평균: `0.8574`

`cross_validate()` 결과의 `test_score`는 최종 테스트 세트 점수가 아니라 **각 폴드의 검증 점수**다. 분류 문제에서는 각 폴드의 클래스 비율을 유지하는 `StratifiedKFold`를 사용한다.

> **이전 질문과 연결 — 훈련 점수와 테스트 점수의 차이는 어떻게 판단하나?**  
> 한 번 분리한 점수만 보면 우연한 샘플 구성의 영향을 받을 수 있다. 여러 폴드의 평균과 점수 분산을 함께 보면 과대적합 여부와 예상 성능을 더 안정적으로 판단할 수 있다.

## 3. 단일 하이퍼파라미터 그리드 서치

`GridSearchCV`는 지정한 후보를 빠짐없이 조합하고, 각 조합을 교차 검증으로 비교한다.

In [12]:
from sklearn.model_selection import GridSearchCV

# 확인할 min_impurity_decrease 후보를 딕셔너리로 정의한다.
params = {
    'min_impurity_decrease': [
        0.0001,
        0.0002,
        0.0003,
        0.0004,
        0.0005
    ]
}

In [13]:
# 각 후보를 5-폴드 교차 검증으로 비교하는 그리드 서치를 만든다.
# n_jobs=-1은 가능한 CPU 코어를 모두 사용한다.
gs = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    params,
    n_jobs=-1
)

In [14]:
# 5개 후보 × 5개 폴드로 성능을 비교한 뒤
# 최적 모델을 전체 훈련 세트로 다시 학습한다.
gs.fit(train_input, train_target)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'min_impurity_decrease': [0.0001, 0.0002, 0.0003,
                                                   0.0004, 0.0005]})

In [15]:
# best_estimator_는 선택된 값으로 전체 훈련 세트를 다시 학습한 모델이다.
dt = gs.best_estimator_

# 이 점수는 교차 검증 점수가 아니라 전체 훈련 세트 정확도이다.
print(dt.score(train_input, train_target))

0.9615162593804117


In [16]:
# 교차 검증 평균이 가장 높았던 하이퍼파라미터를 확인한다.
print(gs.best_params_)

{'min_impurity_decrease': 0.0001}


In [17]:
# 각 min_impurity_decrease 후보의 평균 교차 검증 점수를 확인한다.
# params에 적은 후보 순서와 동일한 순서로 출력된다.
print(gs.cv_results_['mean_test_score'])

[0.86819297 0.86453617 0.86492226 0.86780891 0.86761605]


In [19]:
# best_index_를 이용해 최적 위치의 파라미터 조합을 직접 확인한다.
print(gs.cv_results_['params'][gs.best_index_])

{'min_impurity_decrease': 0.0001}


첫 탐색의 결과는 다음과 같다.

- 최적 값: `min_impurity_decrease=0.0001`
- 후보별 평균 검증 정확도:
  `0.8682`, `0.8645`, `0.8649`, `0.8678`, `0.8676`
- 최적 모델의 전체 훈련 정확도: `0.9615`

`best_estimator_`는 최적 조합을 선택한 뒤 **전체 훈련 세트로 다시 학습된 모델**이다. 따라서 `dt.score(train_input, train_target)`은 교차 검증 점수가 아니라 재학습한 모델의 훈련 점수다.

## 4. 여러 하이퍼파라미터 그리드 서치

이번에는 다음 세 값을 동시에 탐색한다.

- `min_impurity_decrease`: 노드를 나누기 위한 최소 불순도 감소량
- `max_depth`: 트리 최대 깊이
- `min_samples_split`: 노드를 나누기 위해 필요한 최소 샘플 수

In [20]:
# 세 하이퍼파라미터의 후보를 동시에 정의한다.
# 조합 수는 9 × 15 × 10 = 1350개이다.
params = {
    'min_impurity_decrease': np.arange(0.0001, 0.001, 0.0001),
    'max_depth': range(5, 20, 1),
    'min_samples_split': range(2, 100, 10)
}

In [21]:
# 1350개 조합을 각각 5-폴드 교차 검증으로 비교한다.
gs = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    params,
    n_jobs=-1
)
gs.fit(train_input, train_target)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': range(5, 20),
                         'min_impurity_decrease': array([0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007, 0.0008,
       0.0009]),
                         'min_samples_split': range(2, 100, 10)})

In [22]:
# 가장 높은 평균 교차 검증 점수를 만든 조합을 확인한다.
print(gs.best_params_)

{'max_depth': 14, 'min_impurity_decrease': np.float64(0.0004), 'min_samples_split': 12}


In [24]:
# 전체 조합 중 최고 평균 교차 검증 정확도를 확인한다.
print(np.max(gs.cv_results_['mean_test_score']))

0.8683865773302731


후보 조합은 `9 × 15 × 10 = 1350개`이고, 기본 5-폴드이므로 총 `6750회`의 학습·검증이 수행된다.

- 최적 조합:
  `max_depth=14`,
  `min_impurity_decrease=0.0004`,
  `min_samples_split=12`
- 최고 평균 검증 정확도: `0.8684`

> **이전 질문과 연결 — “하이퍼파라미터는 내가 직접 설정해야 하나?”**  
> 사람이 탐색할 파라미터와 범위를 정하고, 그 범위 안의 비교는 검색 도구가 자동으로 수행한다. 범위를 너무 좁게 잡으면 더 나은 값을 놓치고, 너무 넓고 촘촘하게 잡으면 계산량이 급격히 늘어난다.

## 5. 랜덤 서치에 사용할 확률분포

그리드 서치는 모든 조합을 검사하므로 파라미터와 후보가 많아질수록 계산량이 커진다. 랜덤 서치는 지정한 분포에서 일부 조합만 뽑는다.

In [25]:
from scipy.stats import uniform, randint

# 랜덤 서치에서 사용할 연속형·정수형 확률분포를 불러온다.

In [27]:
# 0 이상 10 미만의 정수를 같은 확률로 뽑는 분포를 만든다.
rgen = randint(0, 10)

# 실행할 때마다 다른 정수 10개가 나올 수 있다.
rgen.rvs(10)

array([7, 0, 2, 4, 5, 4, 6, 3, 2, 8])

In [29]:
# 정수 1000개를 뽑아 값별 등장 횟수를 확인한다.
# 각 값이 대략 비슷한 횟수로 선택되는 것을 볼 수 있다.
np.unique(rgen.rvs(1000), return_counts=True)

(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([ 81,  93, 101,  87,  99, 116, 101,  99, 105, 118]))

In [30]:
# 0 이상 1 미만의 실수를 균등하게 뽑는 분포를 만든다.
ugen = uniform(0, 1)

# 실행할 때마다 다른 실수 10개가 나올 수 있다.
ugen.rvs(10)

array([0.69543978, 0.22825689, 0.33671273, 0.33195707, 0.84347914,
       0.5871867 , 0.63820362, 0.46369375, 0.92180494, 0.02459213])

- `randint(a, b)`: `a` 이상 `b` 미만의 정수 추출
- `uniform(a, b)`: `a`에서 시작해 폭 `b`인 구간의 실수 추출

난수 샘플 셀은 다시 실행하면 출력값이 달라질 수 있다.

## 6. 랜덤 서치

`RandomizedSearchCV`는 모든 조합을 만들지 않고 `n_iter=100`개의 조합만 추출한다. 기본 5-폴드이므로 `500회`의 학습·검증으로 탐색한다.

In [31]:
# 랜덤 서치가 값을 뽑을 하이퍼파라미터 분포를 정의한다.
params = {
    'min_impurity_decrease': uniform(0.0001, 0.001),
    'max_depth': randint(20, 50),
    'min_samples_split': randint(2, 25),
    'min_samples_leaf': randint(1, 25)
}

In [32]:
from sklearn.model_selection import RandomizedSearchCV

# 분포에서 100개 조합만 무작위로 뽑아 5-폴드 교차 검증한다.
rs = RandomizedSearchCV(
    DecisionTreeClassifier(random_state=42),
    params,
    n_iter=100,
    n_jobs=-1,
    random_state=42
)
rs.fit(train_input, train_target)

RandomizedSearchCV(estimator=DecisionTreeClassifier(random_state=42),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7b9e478e1640>,
                                        'min_impurity_decrease': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7b9e478e0890>,
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7b9e4818fe00>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7b9e4818e630>},
                   random_state=42)

In [33]:
# 랜덤 서치에서 선택된 최적 하이퍼파라미터를 확인한다.
print(rs.best_params_)

{'max_depth': 39, 'min_impurity_decrease': np.float64(0.00034102546602601173), 'min_samples_leaf': 7, 'min_samples_split': 13}


In [34]:
# 100개 조합 중 최고 평균 교차 검증 정확도를 확인한다.
print(np.max(rs.cv_results_['mean_test_score']))

0.8695428296438884


In [37]:
# 최적 조합으로 전체 훈련 세트를 다시 학습한 모델을 가져온다.
dt = rs.best_estimator_

# 모든 모델 선택이 끝난 뒤 최종 테스트 세트를 한 번 평가한다.
print(dt.score(test_input, test_target))

0.86


결과는 다음과 같다.

- 최적 조합:
  `max_depth=39`,
  `min_impurity_decrease≈0.000341`,
  `min_samples_leaf=7`,
  `min_samples_split=13`
- 최고 평균 검증 정확도: `0.8695`
- 최종 테스트 정확도: `0.8600`

그리드 서치보다 훨씬 적은 조합을 확인하면서 비슷하거나 조금 높은 교차 검증 점수를 얻었다. 모든 선택을 마친 뒤에만 `test_input`을 사용해 최종 성능을 확인했다.

## 7. `splitter='random'` 비교

기본 `splitter='best'`는 각 노드에서 가장 좋은 분할을 찾는다. `splitter='random'`은 무작위 분할 후보 중 하나를 선택해 트리에 무작위성을 추가한다.

In [38]:
from sklearn.model_selection import RandomizedSearchCV

# splitter='random'인 결정 트리도 같은 분포에서 탐색한다.
# 각 노드에서 최적 분할만 찾는 대신 무작위 후보 분할을 사용한다.
rs = RandomizedSearchCV(
    DecisionTreeClassifier(
        splitter='random',
        random_state=42
    ),
    params,
    n_iter=100,
    n_jobs=-1,
    random_state=42
)
rs.fit(train_input, train_target)

RandomizedSearchCV(estimator=DecisionTreeClassifier(random_state=42,
                                                    splitter='random'),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7b9e478e1640>,
                                        'min_impurity_decrease': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7b9e478e0890>,
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7b9e4818fe00>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7b9e4818e630>},
                   random_state=42)

In [39]:
# splitter='random' 모델의 최적 조합, 교차 검증 점수,
# 최종 테스트 정확도를 차례대로 확인한다.
print(rs.best_params_)
print(np.max(rs.cv_results_['mean_test_score']))

dt = rs.best_estimator_
print(dt.score(test_input, test_target))

{'max_depth': 43, 'min_impurity_decrease': np.float64(0.00011407982271508446), 'min_samples_leaf': 19, 'min_samples_split': 18}
0.8458726956392981
0.786923076923077


- 최고 평균 검증 정확도: `0.8459`
- 최종 테스트 정확도: `0.7869`

이 실습에서는 기본 분할보다 성능이 낮았다. 무작위 분할이 항상 성능을 높이는 것은 아니며, 데이터와 다른 설정에 따라 결과가 달라진다.

> **이전 질문과 연결 — “`splitter='random'`으로 하면 엑스트라 트리인가?”**  
> 아니다. 이것은 한 개의 `DecisionTreeClassifier`가 노드 분할을 무작위로 고르게 만든 것이다. 엑스트라 트리는 이런 무작위성이 강한 트리를 여러 개 학습해 결과를 합치는 앙상블 모델이다.

## 정리

```text
테스트 세트를 마지막 평가용으로 분리
→ 검증 세트로 기본 성능 확인
→ 교차 검증으로 분할 우연성 완화
→ GridSearchCV로 정해진 후보 전수 비교
→ RandomizedSearchCV로 넓은 범위 일부 탐색
→ 최적 모델을 전체 훈련 세트로 재학습
→ 테스트 세트로 최종 성능 한 번 평가
```

핵심은 **하이퍼파라미터를 테스트 세트로 고르지 않는 것**이다. 모델 선택은 검증 세트나 교차 검증으로 끝내고, 테스트 세트는 최종 성능 확인에만 사용한다.